# E-Commerce Sales Analysis using Python & Pandas
**Dataset:** UCI Online Retail (2010–2011)  
**Business case:** An online gift retailer wants to understand revenue trends, best-selling products, key markets and customer value.

> Dataset has sales price but no product cost. This project analyses **revenue**, not actual profit. Returns/cancellations are excluded from the main positive-sales analysis and should be monitored separately.

## 1. Setup and data loading

Run this notebook in Google Colab or Jupyter. The Excel file is downloaded from the official UCI dataset source.

In [ ]:
!pip -q install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set_theme(style="whitegrid")

file_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

df = pd.read_excel(file_url)

print("Data loaded successfully")
print("Shape:", df.shape)
print(df.head())

### Understand the dataset

Each row represents one product line within an invoice. `InvoiceNo` beginning with C indicates a cancellation. `UnitPrice` is in GBP (£).

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumeric summary:")
print(df[['Quantity', 'UnitPrice']].describe())

## 2. Data cleaning

Cleaning decisions:
- Remove exact duplicate rows.
- Convert invoice date to datetime and numeric columns to numeric.
- Remove rows without product description or usable date.
- Keep only positive quantities and positive prices for the main sales analysis. This removes returns/cancellations and zero/negative price adjustments from the positive-sales dataset.
- Keep rows with missing CustomerID for overall sales/product/region analysis; create a separate customer dataset for RFM.
- Flag extreme values using IQR. Do not automatically delete every outlier because bulk orders may be genuine.

In [ ]:
data = df.copy()

# Column names ko consistent banate hain
data.columns = data.columns.str.strip()

data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], errors='coerce')
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')
data['UnitPrice'] = pd.to_numeric(data['UnitPrice'], errors='coerce')

before = len(data)

data = data.drop_duplicates()
data = data.dropna(subset=['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate',
                           'Quantity', 'UnitPrice'])

# Cancellation / return rows aur invalid sales values alag rakhne ke bajay
# main sales analysis se exclude kar rahe hain.
cancel_rows = data['InvoiceNo'].astype(str).str.upper().str.startswith('C')
print("Cancellation rows:", cancel_rows.sum())

data = data[~cancel_rows]
data = data[(data['Quantity'] > 0) & (data['UnitPrice'] > 0)]

# CustomerID missing ho sakta hai, sales analysis mein unko abhi retain karenge
data['CustomerID'] = data['CustomerID'].astype('Int64')

data['Revenue'] = data['Quantity'] * data['UnitPrice']

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(data))
print("Rows removed:", before - len(data))
print("Missing CustomerID after cleaning:", data['CustomerID'].isna().sum())

print(data.head())

### Outlier check (IQR method)

We check `Quantity`, `UnitPrice` and `Revenue`. IQR flags unusual records; it does not prove they are errors. For the main revenue totals we retain valid positive transactions, and inspect extreme records separately.

In [ ]:
def iqr_limits(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return low, high

for col in ['Quantity', 'UnitPrice', 'Revenue']:
    low, high = iqr_limits(data[col])
    outlier_count = ((data[col] < low) | (data[col] > high)).sum()
    print(col, " | lower:", round(low, 2), " upper:", round(high, 2),
          " | flagged rows:", outlier_count)

# Highest-value records ko manually inspect kar sakte hain
print("\nTop unusual revenue rows:")
print(data.nlargest(10, 'Revenue')[['InvoiceNo', 'Description', 'Quantity',
                                    'UnitPrice', 'Revenue', 'Country']])

## 3. Feature engineering

Create month, year, weekday and invoice-level details. This makes trend analysis easier.

In [ ]:
data['YearMonth'] = data['InvoiceDate'].dt.to_period('M').astype(str)
data['MonthName'] = data['InvoiceDate'].dt.month_name()
data['DayName'] = data['InvoiceDate'].dt.day_name()
data['Year'] = data['InvoiceDate'].dt.year

print(data[['InvoiceDate', 'YearMonth', 'DayName', 'Revenue']].head())

## 4. Exploratory Data Analysis (EDA)

### 4.1 Monthly revenue trend

In [ ]:
monthly_sales = data.groupby('YearMonth', as_index=False)['Revenue'].sum()

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x='YearMonth', y='Revenue', marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(monthly_sales.sort_values('Revenue', ascending=False).head(5))

### 4.2 Top products by revenue and quantity

In [ ]:
top_products = (data.groupby('Description')
                .agg(Total_Revenue=('Revenue', 'sum'),
                     Units_Sold=('Quantity', 'sum'))
                .sort_values('Total_Revenue', ascending=False)
                .head(10)
                .reset_index())

plt.figure(figsize=(10, 6))
sns.barplot(data=top_products, y='Description', x='Total_Revenue')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

print(top_products)

### 4.3 Top customers by revenue

Customer-level analysis only uses rows with a known CustomerID.

In [ ]:
customer_sales = data.dropna(subset=['CustomerID']).copy()

top_customers = (customer_sales.groupby('CustomerID')['Revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10)
                 .reset_index())

plt.figure(figsize=(10, 5))
sns.barplot(data=top_customers, x='CustomerID', y='Revenue')
plt.title('Top 10 Customers by Revenue')
plt.xlabel('Customer ID')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(top_customers)

### 4.4 Region / country-wise revenue

In [ ]:
country_sales = (data.groupby('Country')['Revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10)
                 .reset_index())

plt.figure(figsize=(10, 5))
sns.barplot(data=country_sales, y='Country', x='Revenue')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

print(country_sales)

### 4.5 Sales by day of week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

weekday_sales = data.groupby('DayName')['Revenue'].sum().reindex(day_order)

plt.figure(figsize=(9, 4))
sns.barplot(x=weekday_sales.index, y=weekday_sales.values)
plt.title('Revenue by Day of Week')
plt.xlabel('Day')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Customer segmentation – simple RFM

- **Recency:** Days since the customer's last purchase. Lower is better.
- **Frequency:** Number of distinct invoices. Higher is better.
- **Monetary:** Total revenue from the customer. Higher is better.

The snapshot date is one day after the latest transaction in the cleaned dataset. This avoids using today's date for a historical dataset.

In [ ]:
snapshot_date = customer_sales['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = customer_sales.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

print(rfm.head())
print("Customers in RFM:", len(rfm))

In [ ]:
# Quantile scores: 1 to 4
# Recency mein kam days = better score, isliye reverse scoring.
rfm['R_score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4,
                         labels=[4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4,
                         labels=[1, 2, 3, 4]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4,
                         labels=[1, 2, 3, 4]).astype(int)

rfm['RFM_Score'] = (rfm['R_score'].astype(str) +
                    rfm['F_score'].astype(str) +
                    rfm['M_score'].astype(str))

def customer_segment(row):
    if row['R_score'] >= 3 and row['F_score'] >= 3 and row['M_score'] >= 3:
        return 'Best Customers'
    elif row['R_score'] >= 3 and row['F_score'] >= 2:
        return 'Loyal / Active'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Lost / Inactive'
    else:
        return 'Potential Customers'

rfm['Segment'] = rfm.apply(customer_segment, axis=1)

segment_summary = (rfm.groupby('Segment')
                   .agg(Customers=('CustomerID', 'count'),
                        Revenue=('Monetary', 'sum'),
                        Avg_Recency=('Recency', 'mean'))
                   .sort_values('Revenue', ascending=False))

print(segment_summary)

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=segment_summary.index, y=segment_summary['Customers'])
plt.title('Customer Count by RFM Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 6. Business analysis and insights

Run this section after the analysis above. Values are calculated from the cleaned dataset, so do not copy example numbers from another project or dataset version.

In [ ]:
total_revenue = data['Revenue'].sum()
total_orders = data['InvoiceNo'].nunique()
total_units = data['Quantity'].sum()
unique_products = data['StockCode'].nunique()
unique_customers = customer_sales['CustomerID'].nunique()
avg_order_value = total_revenue / total_orders if total_orders else 0

best_month = monthly_sales.loc[monthly_sales['Revenue'].idxmax()]
best_country = country_sales.iloc[0]
best_product = top_products.iloc[0]

print("----- BUSINESS KPI SUMMARY -----")
print("Total Revenue (£):", round(total_revenue, 2))
print("Unique Invoices:", total_orders)
print("Units Sold:", int(total_units))
print("Unique Products:", unique_products)
print("Identified Customers:", unique_customers)
print("Average Revenue per Invoice (£):", round(avg_order_value, 2))

print("\n----- QUICK INSIGHTS -----")
print("Highest revenue month:", best_month['YearMonth'],
      "| Revenue (£):", round(best_month['Revenue'], 2))
print("Top country:", best_country['Country'],
      "| Revenue (£):", round(best_country['Revenue'], 2))
print("Top product by revenue:", best_product['Description'],
      "| Revenue (£):", round(best_product['Total_Revenue'], 2))
print("Largest RFM segment by customer count:",
      segment_summary['Customers'].idxmax())

## 7. Actionable business recommendations

Use the actual outputs above to validate these recommendations before presenting:
1. **Protect best-selling products:** Keep high-revenue products in stock and monitor stock-outs.
2. **Plan for seasonal demand:** Increase inventory and campaign preparation before the strongest revenue months. Check whether the final month is a partial month.
3. **Retain high-value customers:** Offer early access, loyalty benefits or personalised bundles to Best Customers.
4. **Reactivate at-risk customers:** Send targeted win-back offers to customers with high past frequency/value but long recency.
5. **Localise country campaigns:** Prioritise marketing in high-revenue countries, while checking delivery cost and return rates before increasing spend.
6. **Improve data and margin visibility:** Track returns separately and add product cost, shipping and discount data. Then calculate gross profit and margin instead of treating revenue as profit.

## 8. Export cleaned data and RFM output

In [ ]:
data.to_csv('cleaned_ecommerce_sales.csv', index=False)
rfm.to_csv('customer_rfm_segments.csv', index=False)

print("Files exported successfully")

## 9. Short project report

**Project:** E-Commerce Sales Analysis using Python and Pandas

**Objective:** Analyse historical retail transactions to understand revenue trends, product performance, geographic performance and customer value.

**Tools:** Python, Pandas, NumPy, Matplotlib and Seaborn.

**Work performed:** Loaded the public UCI Online Retail dataset, checked missing values and duplicates, removed cancellation/return and invalid sales rows from the positive-sales view, engineered date and revenue columns, explored monthly/product/customer/country trends, flagged outliers using IQR and created simple RFM customer segments.

**Business value:** The analysis helps the retailer identify important products and markets, plan for seasonal demand and design customer retention campaigns.

**Limitations:** The dataset covers a historical period (December 2010 to December 2011), and it does not contain product cost, shipping cost or discount details. Therefore, this analysis reports revenue and cannot establish actual profit or current customer behaviour.